# Artificial Intelligence for Whole Slide Imaging: <br/> Application to Acute Lymphoblastic Leukemia Detection
# I2MTC 2026 – Tutorial

**Dataset ALL-IDB1**

This dataset is composed of 108 images collected during September, 2005. It contains about 39000 blood elements, where the lymphocytes has been labeled by expert oncologists. The images are taken with different magnifications of the microscope ranging from 300x to 500x.

Example:

<img src="WSIs/Im001_1.jpg" alt="Alt text" width="600">

The tutorial will focus on giving the tools to realize design, train, and apply Deep Learning models for Acute Lymphoblastic Leukemia. The detection of the disease will be achieved through an effective processing of Whole Slide Images of blood tissues, resulting in the classification of white blood cells as either “normal” or “lymphoblasts”. The tutorial will also discuss the relevant datasets, models, and classification metrics.

## Part 1. Handle Whole Slide Images

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# IMPORT
from __future__ import annotations
import os
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Tuple
import numpy as np
import pandas as pd
from PIL import Image

### Display Whole Slide Images

Let's display one of the whole slide images (WSI).

In [ ]:
# directory of WSI
dirWSIs = Path('./WSIs/')
ext = 'jpg'

# extract list of files
files = sorted(dirWSIs.glob(f"*.{ext}"))
# get the first file and open
img_path = next(enumerate(files))
img = Image.open(img_path[1])

# display size and img
print("Size of the image: {0}".format(img.size))
display(img)

### WSI annotations

In our case, for each WSI we have a list of centroids, each describing whether the white blood cell (WBC) is a 'blast' or 'normal'.

Let's define a function to import the information.

In [ ]:
# read_blast_normal_file
def read_blast_normal_file(path: Path) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Python version of readBlastNormalFile.m.

    Expected format:
      blasts
      x y
      x y
      ...
      normal
      x y
      ...
    """
    x_blasts: List[float] = []
    y_blasts: List[float] = []
    x_normal: List[float] = []
    y_normal: List[float] = []

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        lines = [ln.strip() for ln in f.readlines() if ln.strip() != ""]

    if not lines:
        return np.array([]), np.array([]), np.array([]), np.array([])

    idx = 0
    if lines[idx].lower() == "blasts":
        idx += 1
        while idx < len(lines):
            if lines[idx].lower() == "normal":
                idx += 1
                break
            parts = lines[idx].split()
            if len(parts) >= 2:
                x_blasts.append(float(parts[0]))
                y_blasts.append(float(parts[1]))
            idx += 1

    while idx < len(lines):
        parts = lines[idx].split()
        if len(parts) >= 2:
            x_normal.append(float(parts[0]))
            y_normal.append(float(parts[1]))
        idx += 1

    return (
        np.array(x_blasts, dtype=float),
        np.array(y_blasts, dtype=float),
        np.array(x_normal, dtype=float),
        np.array(y_normal, dtype=float),
    )

We need a function for display purposes.

In [ ]:
from typing import Iterable, Optional, Tuple, Union
from pathlib import Path
import matplotlib.pyplot as plt

# overlay_centroids
def overlay_centroids(
    image: Union[str, Path, Image.Image, np.ndarray],
    x_blasts: Iterable[float], y_blasts: Iterable[float], x_normals: Iterable[float], y_normals: Iterable[float],
    *,
    one_based: bool = False, show_labels: bool = False,
    label_fontsize: int = 8,
    blast_marker: str = "x", normal_marker: str = "o",
    blast_color: str = "red", normal_color: str = "blue",
    blast_markersize: int = 8, normal_markersize: int = 6,
    alpha: float = 0.9,
    figsize: Tuple[float, float] = (10, 10),
    save_path: Optional[Union[str, Path]] = None,
    dpi: int = 150,
) -> Tuple[plt.Figure, plt.Axes]:
    """
    Overlay centroid coordinates on an image.

    Args:
        image: path to image file or numpy array (H x W) or (H x W x C).
        x_blasts, y_blasts, x_normals, y_normals: iterables of numeric coordinates.
            Coordinates are expected as (x = column, y = row) (i.e., MATLAB convention).
        one_based: if True, subtracts 1 from coordinates (MATLAB -> Python conversion).
        show_labels: if True, annotate each point with its index.
        label_fontsize: font size for annotations.
        blast_marker / normal_marker: marker styles for blasts / normals.
        blast_color / normal_color: colors.
        blast_markersize / normal_markersize: marker sizes.
        alpha: marker transparency.
        figsize: matplotlib figure size.
        save_path: if provided, save the resulting image to this path.
        dpi: save dpi if saving.

    Returns:
        (fig, ax) tuple for further manipulation or saving.
    """
    # load image
    if isinstance(image, (str, Path)):
        img = Image.open(str(image)).convert("RGB")
        img_np = np.asarray(img)

    elif isinstance(image, Image.Image):
        img_np = np.asarray(image.convert("RGB"))

    elif isinstance(image, np.ndarray):
        img_np = image.copy()
        # ensure RGB for display
        if img_np.ndim == 2:
            img_np = np.stack([img_np] * 3, axis=-1)
        elif img_np.ndim == 3 and img_np.shape[2] == 1:
            img_np = np.concatenate([img_np] * 3, axis=2)
    else:
        raise ValueError("image must be a path or numpy array")

    H, W = img_np.shape[:2]

    # prepare coords
    x_b = np.asarray(list(x_blasts), dtype=float) if x_blasts is not None else np.array([])
    y_b = np.asarray(list(y_blasts), dtype=float) if y_blasts is not None else np.array([])
    x_n = np.asarray(list(x_normals), dtype=float) if x_normals is not None else np.array([])
    y_n = np.asarray(list(y_normals), dtype=float) if y_normals is not None else np.array([])

    # optional one-based -> zero-based conversion
    if one_based:
        x_b = x_b - 1.0
        y_b = y_b - 1.0
        x_n = x_n - 1.0
        y_n = y_n - 1.0

    # Clip coordinates (optional) so they are inside image bounds
    # But do not discard: just clamp for plotting.
    x_b = np.clip(x_b, 0, W - 1)
    x_n = np.clip(x_n, 0, W - 1)
    y_b = np.clip(y_b, 0, H - 1)
    y_n = np.clip(y_n, 0, H - 1)

    # Matplotlib expects (x, y) with origin at top-left if we invert y-axis.
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img_np)
    ax.set_xlim(-0.5, W - 0.5)
    ax.set_ylim(H - 0.5, -0.5)  # flip y to match image coordinates (top-left origin)
    ax.set_xlabel("x (columns)")
    ax.set_ylabel("y (rows)")
    ax.set_title("Centroid overlay: blasts (red x) and normals (blue o)")

    # plot blasts
    if x_b.size > 0:
        ax.scatter(x_b, y_b, marker=blast_marker, s=blast_markersize**2,
                   c=blast_color, label="blast", alpha=alpha, linewidths=1.0)
        if show_labels:
            for i, (xx, yy) in enumerate(zip(x_b, y_b), start=1):
                ax.text(xx + 2, yy - 2, f"B{i}", color=blast_color,
                        fontsize=label_fontsize, weight="bold", alpha=0.9)

    # plot normals
    if x_n.size > 0:
        ax.scatter(x_n, y_n, marker=normal_marker, s=normal_markersize**2,
                   facecolors='none', edgecolors=normal_color, label="normal", alpha=alpha, linewidths=1.0)
        if show_labels:
            for i, (xx, yy) in enumerate(zip(x_n, y_n), start=1):
                ax.text(xx + 2, yy - 2, f"N{i}", color=normal_color,
                        fontsize=label_fontsize, weight="bold", alpha=0.9)

    ax.legend(loc="upper right")
    ax.set_axis_off()  # optional: hide axes for cleaner image

    plt.tight_layout()

    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(save_path), dpi=dpi, bbox_inches="tight")
        print(f"Saved overlay image to: {save_path}")

    return fig, ax

Let's now import one of those files and display the information.

In [ ]:
# directory of information
dirPositions = dirWSIs / 'positions'
extInfo = 'xyc'
# extract list of files
filesInfo = sorted(dirPositions.glob(f"*.{extInfo}"))
# get the first file and open
info_path = next(enumerate(filesInfo))
x_blasts, y_blasts, x_normals, y_normals = read_blast_normal_file(info_path[1])

# display
fig, ax = overlay_centroids(img, x_blasts, y_blasts, x_normals, y_normals, show_labels=False,
                            blast_markersize=14, normal_markersize=20, normal_color='yellow')
plt.show()

### Split WSI into patches

Using the information we have, we need to:
* split the WSI in patches;
* assign each patch the corresponding label(s).

First, we need to decide how we want to split into patches, by choosing:
* patch size;
* how much we want to overlap.

Experimentally, we found good values using patches with size 256x256 pixels, with 25% overlap between patches.

In [ ]:
PATCH_SIZE = 256    # patch size
OVERLAP_PCT = 0.25  # overlap percentage (0.25 = 25%)

Let's now define a function that computes the grid, based on patch size and overlap.

In [ ]:
# compute_patch_grid
def compute_patch_grid(h: int, w: int, patch_size: int, overlap_pct: float) -> List[Tuple[int, int]]:
    """
    Returns a list of (row_start, col_start) for each patch.
    """
    step = int(round(patch_size * (1 - overlap_pct)))  # blockOffsets in MATLAB
    if step <= 0:
        raise ValueError("Invalid overlap_pct resulting in non-positive step.")

    starts = []
    # Exclude incomplete blocks: start must allow a full patch inside image
    for r in range(0, h - patch_size + 1, step):
        for c in range(0, w - patch_size + 1, step):
            starts.append((r, c))
    return starts

Now we can use the patch grid to split the image into patches.
We also want to assign the label to each patch.

To do so, we need to check whether each patch contains a centroid for a blast or a normal WBC. Given that WBC can be irregularly shaped, let's define a bit of a tolerance.

In [ ]:
PIXEL_TOLL = 5  # tolerance (in px)

If we need to check whether each patch contains centroids for blast or normal WBC, we need to keep track of the original position of each patch.
We can do it with the class <code>PatchInfo</code>.

In [ ]:
# Patch "info"
@dataclass
class PatchInfo:
    # Mimic MATLAB info.Start / info.End (row, col) style
    Start: Tuple[int, int]  # (row_start, col_start)
    End: Tuple[int, int]    # (row_end, col_end)
    BlockSize: Tuple[int, int]

And we need another function to just check that the tolerance does not yield values outside the boundaries of the cell.

In [ ]:
# bound
def bound(x: float | int, bl: float | int, bu: float | int) -> float | int:
    """Return bounded value clipped between bl and bu (bound.m)."""
    return min(max(x, bl), bu)

We can extract the patches. The info of each patch is then checked to assign the label. The labels are not mutually exclusive: a patch can contain both a normal WBC and a blast.

In [ ]:
import pickle

def save_patchinfos_pickle(path: Path, patchinfos: list[PatchInfo]):
    with open(path, "wb") as f:
        pickle.dump(patchinfos, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_patchinfos_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)

# extract_patches_for_image
def extract_patches_for_image(img_path: Path, labels_dir: Path) -> pd.DataFrame:
    """
    Process one image: extract patches, label them, save patches + info,
    return rows for the CSV table.
    """
    # Load image (PIL -> RGB/gray preserved)
    img = Image.open(img_path)
    img_np = np.array(img)
    if img_np.ndim == 2:
        h, w = img_np.shape
    else:
        h, w = img_np.shape[:2]

    filename_img = img_path.name
    filename_noext = img_path.stem

    # Label file name: replace ext with 'xyc' like MATLAB strrep
    label_file = labels_dir / f"{filename_noext}.xyc"
    x_blasts, y_blasts, x_normals, y_normals = read_blast_normal_file(label_file)

    # Patch grid
    starts = compute_patch_grid(h, w, PATCH_SIZE, OVERLAP_PCT)

    rows = []

    # Tracking maxima like MATLAB (optional)
    max_blasts = -1
    max_normals = -1

    for patch_idx, (r0, c0) in enumerate(starts, start=1):
        # "matlab_like": store 0-based starts (common in many Python libs),
        # while still keeping the same comparisons & centroid computation structure.
        start_row = r0
        start_col = c0
        end_row = start_row + PATCH_SIZE - 1
        end_col = start_col + PATCH_SIZE - 1

        info = PatchInfo(
            Start=(int(start_row), int(start_col)),
            End=(int(end_row), int(end_col)),
            BlockSize=(PATCH_SIZE, PATCH_SIZE),
        )

        # Extract patch (Python slicing is 0-based)
        patch = img_np[r0 : r0 + PATCH_SIZE, c0 : c0 + PATCH_SIZE].copy()

        blast_present = 0
        normal_present = 0
        count_blasts = 0
        count_normals = 0
        centroids_blasts: List[float] = []
        centroids_normals: List[float] = []

        # Coordinate checks
        # blasts inside (Start..End) expanded by PIXEL_TOLL
        for xb, yb in zip(x_blasts, y_blasts):
            c1 = xb >= (info.Start[1] - PIXEL_TOLL)
            c2 = xb <= (info.End[1] + PIXEL_TOLL)
            c3 = yb >= (info.Start[0] - PIXEL_TOLL)
            c4 = yb <= (info.End[0] + PIXEL_TOLL)
            if c1 and c2 and c3 and c4:
                blast_present = 1
                count_blasts += 1
                # MATLAB: bound(x - Start(2), 1, sizePatches)
                # MATLAB: bound(y - Start(1), 1, sizePatches)
                cx = bound(xb - info.Start[1], 1, PATCH_SIZE)
                cy = bound(yb - info.Start[0], 1, PATCH_SIZE)
                centroids_blasts.extend([cx, cy])

        for xn, yn in zip(x_normals, y_normals):
            c1 = xn >= (info.Start[1] - PIXEL_TOLL)
            c2 = xn <= (info.End[1] + PIXEL_TOLL)
            c3 = yn >= (info.Start[0] - PIXEL_TOLL)
            c4 = yn <= (info.End[0] + PIXEL_TOLL)
            if c1 and c2 and c3 and c4:
                normal_present = 1
                count_normals += 1
                cx = bound(xn - info.Start[1], 1, PATCH_SIZE)
                cy = bound(yn - info.Start[0], 1, PATCH_SIZE)
                centroids_normals.extend([cx, cy])

        max_blasts = max(max_blasts, count_blasts)
        max_normals = max(max_normals, count_normals)

        filename_patch = f"{filename_noext}_patch_{patch_idx}.tif"
        out_patch_path = dirPatches / filename_patch
        Image.fromarray(patch).save(out_patch_path)
        filename_patch_info = f"{filename_noext}_patch_{patch_idx}.dat"
        out_patch_info = dirInfo / filename_patch_info
        save_patchinfos_pickle(out_patch_info, info)

        # Table centroid formatting like MATLAB: num2str(vector) or 'N/A'
        blasts_str = "N/A" if len(centroids_blasts) == 0 else " ".join(map(lambda v: f"{v:g}", centroids_blasts))
        normals_str = "N/A" if len(centroids_normals) == 0 else " ".join(map(lambda v: f"{v:g}", centroids_normals))

        rows.append(
            {
                "Filename": filename_patch,
                "White blood cell probable ALL lymphoblast": int(blast_present),
                "White blood cell NOT a probable lymphoblast": int(normal_present),
                "Centroids blasts": blasts_str,
                "Centroids NOT blasts": normals_str,
            }
        )

    return pd.DataFrame(rows)

Let's now run the main().

In [ ]:
# dir patches
dirPatches = Path('./') / f'patches_{PATCH_SIZE}_overlap_{OVERLAP_PCT}_toll_{PIXEL_TOLL}'
os.makedirs(dirPatches, exist_ok=True)
dirInfo = Path('./') / f'patches_{PATCH_SIZE}_overlap_{OVERLAP_PCT}_toll_{PIXEL_TOLL}' / 'info'
os.makedirs(dirInfo, exist_ok=True)

# Main

# List images
files = sorted(dirWSIs.glob(f"*.{ext}"))
if not files:
    raise FileNotFoundError(f"No images found in {dirWSIs} with extension .{ext}")

# csv name
csv_name = f"ALL_IDB1_patches_{PATCH_SIZE}_overlap_{OVERLAP_PCT}_toll_{PIXEL_TOLL}.csv"
out_csv = dirPatches / csv_name

if 0:
    all_rows = []

    for img_path in files:
        print(img_path.name)
        df_img = extract_patches_for_image(img_path, dirPositions)
        all_rows.append(df_img)

    df_all = pd.concat(all_rows, ignore_index=True)

    # Write CSV like MATLAB
    df_all.to_csv(out_csv, index=False)
    print(f"\nWrote CSV: {out_csv}")
    print(f"Wrote patches to: {dirPatches}")


### Display patches

Let's now display the patches we created.

In [ ]:
from typing import Optional, Sequence, Tuple, List, Union, Dict, Any
import math
import re
import pandas as pd

def _natural_key(s: str):
    import re
    parts = re.split(r'(\d+)', s)
    key = []
    for p in parts:
        if p.isdigit():
            key.append(int(p))
        else:
            key.append(p.lower())
    return key

def _parse_centroid_string(s: str) -> List[Tuple[float, float]]:
    """Parse centroid string like '12 34 56 78' -> [(12,34),(56,78)]. Returns [] on 'N/A' or empty."""
    if s is None:
        return []
    s = str(s).strip()
    if s == "" or s.upper() == "N/A":
        return []
    parts = re.split(r'[\s,;]+', s)
    nums = []
    for p in parts:
        if p == '':
            continue
        try:
            nums.append(float(p))
        except ValueError:
            # skip tokens that are not numbers
            pass
    pts = []
    for i in range(0, len(nums) - 1, 2):
        pts.append((nums[i], nums[i + 1]))
    return pts

def mosaic_with_centroids(
    patches_dir: Union[str, Path],
    csv_path: Union[str, Path],
    *,
    wsi_id: Optional[str] = None,
    pattern: Optional[str] = None,
    patch_exts: Sequence[str] = (".tif", ".png", ".jpg", ".jpeg"),
    order: str = "windows",  # "windows" (natural) or "spatial" (parses coordinates from filenames)
    cols: Optional[int] = None,
    thumb_size: Tuple[int, int] = (128, 128),
    max_patches: Optional[int] = None,
    padding: int = 2,
    bg_color: Tuple[int, int, int] = (255, 255, 255),
    show_axes: bool = False,
    show: bool = True,
    save_path: Optional[Union[str, Path]] = None,
    csv_filename_col: str = "Filename",
    centroids_blasts_col: str = "Centroids blasts",
    centroids_normals_col: str = "Centroids NOT blasts",
    one_based: bool = True,
    show_labels: bool = False,
    label_fontsize: int = 6,
):
    """
    Create mosaic of patches and overlay centroids from CSV.

    Returns (mosaic_pil_image, matplotlib_figure)

    Important:
      - Centroids are expected to be coordinates relative to each patch (x=col, y=row).
      - If centroids were produced in MATLAB, set one_based=True.
    """
    patches_dir = Path(patches_dir)
    csv_path = Path(csv_path)

    if not patches_dir.exists():
        raise FileNotFoundError(f"{patches_dir} not found")
    if not csv_path.exists():
        raise FileNotFoundError(f"{csv_path} not found")

    # read CSV
    df = pd.read_csv(csv_path, dtype=str).fillna("")  # keep as strings for parsing
    # normalize filename column to string
    if csv_filename_col not in df.columns:
        raise ValueError(f"CSV does not contain filename column '{csv_filename_col}'. Columns: {list(df.columns)}")

    # gather files
    if pattern:
        files = sorted(patches_dir.glob(pattern))
    elif wsi_id is not None:
        files = sorted(p for p in patches_dir.iterdir() if p.is_file() and wsi_id in p.name)
    else:
        files = []
        for ext in patch_exts:
            files.extend(list(patches_dir.glob(f"*{ext}")))
        files = sorted(set(files), key=lambda p: p.name)

    if not files:
        raise FileNotFoundError("No patch files found with the given filters.")

    if max_patches is not None:
        files = files[:max_patches]

    # natural/windows ordering
    files = sorted(files, key=lambda p: _natural_key(p.name))

    # decide columns/rows
    n = len(files)
    if cols is None:
        cols = int(math.ceil(math.sqrt(n)))
    cols = max(1, cols)
    rows = int(math.ceil(n / cols))

    thumb_w, thumb_h = thumb_size
    pad = int(padding)
    canvas_w = cols * thumb_w + (cols + 1) * pad
    canvas_h = rows * thumb_h + (rows + 1) * pad

    # We'll display with Matplotlib. Prepare figure sized proportionally.
    fig_w = min(16, canvas_w / 100 * 1.2)
    fig_h = min(16, canvas_h / 100 * 1.2)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    # Build an RGB canvas (we'll paste thumbnails visually via imshow of assembled PIL mosaic later)
    mosaic = Image.new("RGB", (canvas_w, canvas_h), color=bg_color)

    # We'll collect coordinates (for plotting markers over the mosaic) in mosaic (canvas) coordinates
    scatter_blast_x = []
    scatter_blast_y = []
    scatter_normal_x = []
    scatter_normal_y = []
    scatter_blast_labels = []
    scatter_normal_labels = []

    for idx, fp in enumerate(files):
        try:
            im = Image.open(fp)
            if im.mode != "RGB":
                im = im.convert("RGB")
            orig_w, orig_h = im.size
            # create a thumbnail copy
            thumb = im.copy()
            thumb.thumbnail((thumb_w, thumb_h), Image.LANCZOS)
        except Exception as e:
            print(f"Skipping {fp.name} ({e})")
            continue

        # compute top-left position on canvas
        row = idx // cols
        col = idx % cols
        x0 = pad + col * (thumb_w + pad)  # left
        y0 = pad + row * (thumb_h + pad)  # top

        # center small thumbnails inside slot
        x = x0 + max(0, (thumb_w - thumb.width) // 2)
        y = y0 + max(0, (thumb_h - thumb.height) // 2)

        mosaic.paste(thumb, (int(x), int(y)))

        # find corresponding CSV row: try exact filename then stem
        fname = fp.name
        stem = fp.stem
        matched = df[df[csv_filename_col] == fname]
        if matched.empty:
            matched = df[df[csv_filename_col] == stem]
        if matched.empty:
            # try contains
            matched = df[df[csv_filename_col].str.contains(stem, na=False)]
        if matched.empty:
            # no data for this patch: continue
            continue

        # if multiple matches take the first
        row_series = matched.iloc[0]
        str_blasts = row_series.get(centroids_blasts_col, "")
        str_normals = row_series.get(centroids_normals_col, "")

        pts_b = _parse_centroid_string(str_blasts)
        pts_n = _parse_centroid_string(str_normals)

        # If centroids appear to be in (cx, cy) in MATLAB 1-based coords relative to patch:
        # Map them to thumbnail/canvas coordinates:
        # scale_x = thumb.width / orig_w  (if orig_w is patch width)
        # scale_y = thumb.height / orig_h
        # For safety, if orig_w is zero or missing, fall back to thumb_size scale = 1
        if orig_w > 0 and orig_h > 0:
            scale_x = thumb.width / orig_w
            scale_y = thumb.height / orig_h
        else:
            scale_x = scale_y = 1.0

        # convert and append to mosaic coords
        for i_p, (cx, cy) in enumerate(pts_b, start=1):
            # MATLAB 1-based -> convert
            if one_based:
                cx_adj = cx - 1.0
                cy_adj = cy - 1.0
            else:
                cx_adj = cx
                cy_adj = cy
            # scaled position inside thumbnail
            px = x + cx_adj * scale_x
            py = y + cy_adj * scale_y
            scatter_blast_x.append(px)
            scatter_blast_y.append(py)
            if show_labels:
                scatter_blast_labels.append(f"B{i_p}")

        for i_p, (cx, cy) in enumerate(pts_n, start=1):
            if one_based:
                cx_adj = cx - 1.0
                cy_adj = cy - 1.0
            else:
                cx_adj = cx
                cy_adj = cy
            px = x + cx_adj * scale_x
            py = y + cy_adj * scale_y
            scatter_normal_x.append(px)
            scatter_normal_y.append(py)
            if show_labels:
                scatter_normal_labels.append(f"N{i_p}")

    # Show mosaic as background
    ax.imshow(np.asarray(mosaic))
    # Plot blasts and normals over it
    if scatter_blast_x:
        ax.scatter(scatter_blast_x, scatter_blast_y, marker="x", s=30, c="red", label="blast", linewidths=1.2)
        if show_labels:
            for lab, xx, yy in zip(scatter_blast_labels, scatter_blast_x, scatter_blast_y):
                ax.text(xx + 2, yy - 2, lab, color="red", fontsize=label_fontsize, weight="bold")

    if scatter_normal_x:
        ax.scatter(scatter_normal_x, scatter_normal_y, facecolors="none", edgecolors="blue",
                   marker="o", s=30, label="normal", linewidths=1.0)
        if show_labels:
            for lab, xx, yy in zip(scatter_normal_labels, scatter_normal_x, scatter_normal_y):
                ax.text(xx + 2, yy - 2, lab, color="blue", fontsize=label_fontsize, weight="bold")

    ax.set_xlim(0, canvas_w - 1)
    ax.set_ylim(canvas_h - 1, 0)  # flip y so top-left origin matches image coordinates
    if not show_axes:
        ax.axis("off")
    else:
        ax.set_xlabel("x (cols)")
        ax.set_ylabel("y (rows)")

    ax.set_title(f"Mosaic with centroids — {len(files)} patches")
    ax.legend(loc="upper right")

    plt.tight_layout()
    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        mosaic.save(save_path)
        print(f"Saved mosaic image to {save_path}")

    if show:
        plt.show()

    return mosaic, fig

In [ ]:
mosaic, fig = mosaic_with_centroids(
    patches_dir=dirPatches,
    csv_path=out_csv,
    wsi_id="Im001",              # optional substring to filter files
    thumb_size=(128, 128),
    cols=8,  #6/8
    padding=4,
    one_based=True,              # set True if centroids came from MATLAB
    show_labels=False,
)

## Part 2. Model

We need now to train our model to classify each patch and decide whether it contains a normal WBC, a blast, both, or nothing.

To achieve better results, we will use a CNN pretrained on a histopathology database, as described here:

A. Genovese, V. Piuri, and F. Scotti, "ALL-IDB Patches: Whole slide imaging for Acute Lymphoblastic Leukemia detection using Deep Learning", in Proc. of the IEEE Int. Conf. on Acoustics Speech and Signal Processing Workshops (ICASSPW 2023), Rhodes Island, Greece, June 4-10, 2023, pp. 1-5. ISBN: 979-8-3503-0261-5. [DOI: 10.1109/ICASSPW59220.2023.10193429]

In [ ]:
# import
from torchvision import models
from torchvision import transforms
from torchvision import datasets
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import shutil

import util
import functions
from classes.classesADP import classesADP
from classes.classesALL import classesALL
from modelGeno.resnet_geno import resnet18_orth_mtl
from modelGeno.resnet_geno import resnet34_orth_mtl

cuda = True if torch.cuda.is_available() else False
print("Cuda is {0}".format(cuda))
torch.cuda.empty_cache()

**Cross-validation**

Let's divide our dataset into folds.

We use the class info of the CSV to get indexes.

In [ ]:
# get information about classes from the CSV
csvFileFull = dirPatches / csv_name
classVec, fileNameVec, columnNames = util.getAllClassesVec(classesALL, csvFileFull, True)

# params
nFolds = 5
log = True
r = 0

# get indexes for cross-validation
indexes = util.getIndexesCrossValMultiLab(classVec, nFolds)

Then we can split the dataset.

In [ ]:
# first, we convert it to datastore format
extOrig = 'tif'
extNew = 'png'
dirDbTest = './datastore/'
# dLabel = util.dbToDataStore(dirPatches, dirDbTest, extOrig, extNew, log)
dLabel = 'dummyLabel'

# define folds
list_fold = list(range(0, nFolds))
folds = {}
fold_test = (nFolds-1)-r
folds['test'] = [fold_test]
# fold val
if fold_test > 1:
    fold_val = fold_test-1
else:
    fold_val = fold_test+1
folds['val'] = [fold_val]
# fold train - remaining
list_fold_train = list_fold
list_fold_train.remove(fold_test)
list_fold_train.remove(fold_val)
folds['train'] = list_fold_train

# directory we use to keep split dataset
dirOutTrainTest = './datastore_trainTest/'

if 0:
    # clean
    if os.path.exists(dirOutTrainTest):
        shutil.rmtree(dirOutTrainTest)
    # create
    if not os.path.exists(dirOutTrainTest):
        os.mkdir(dirOutTrainTest)
        os.mkdir(os.path.join(dirOutTrainTest, 'train'))
        os.mkdir(os.path.join(dirOutTrainTest, 'val'))
        os.mkdir(os.path.join(dirOutTrainTest, 'test'))
        os.mkdir(os.path.join(dirOutTrainTest, 'train', dLabel))
        os.mkdir(os.path.join(dirOutTrainTest, 'val', dLabel))
        os.mkdir(os.path.join(dirOutTrainTest, 'test', dLabel))
    # split db
    print('Splitting DB...')
    util.splitDBTrainValTest(indexes, os.path.join(dirDbTest, dLabel), dirOutTrainTest, fileNameVec, extNew, dLabel, folds)
    # end define folds

# get labels
classVecPart = {}
fileNameVecPart = {}
classVecPart['train'], fileNameVecPart['train'] = util.extractLabels(os.path.join(dirOutTrainTest, 'train', dLabel), fileNameVec, extOrig, classVec)
classVecPart['val'], fileNameVecPart['val'] = util.extractLabels(os.path.join(dirOutTrainTest, 'val', dLabel), fileNameVec, extOrig, classVec)
classVecPart['test'], fileNameVecPart['test'] = util.extractLabels(os.path.join(dirOutTrainTest, 'test', dLabel), fileNameVec, extOrig, classVec)

**Load model**

Now we can load the model.

Let's define all the parameters of the models.

In [ ]:
# directory pretrained models
dirPretrainedModels = './pretrained_nets/'
# params
num_iterations = 1  # (should be equal to nfolds)

In [ ]:
# define all models we want to try
modelNamesAll = list()
modelNamesAll.append({'name': 'resnet18', 'sizeFeatures': 512})
# modelNamesAll.append({'name': 'resnet34', 'sizeFeatures': 512})
modelData = next(enumerate(modelNamesAll))[1]
print("Model: {0}".format(modelData['name']))

# train modes. ADP = model pretrained on histopathology database
trainModes = []
trainModes.append('adp')
trainMode = next(enumerate(trainModes))[1]
print("Train mode: {0}".format(trainMode))

We load the models and then change the last layer.

In [ ]:
# init
dataset_sizes = {}
accuracyALL = np.zeros(num_iterations)

# load model
if modelData['name'] == 'resnet18':
    currentModel = resnet18_orth_mtl(pretrained=False)
if modelData['name'] == 'resnet34':
    currentModel = resnet34_orth_mtl(pretrained=False)

# change last layer, so we can load results
# Multi-task learning
# fc1
new_classifier1 = nn.Linear(modelData['sizeFeatures'], classesADP[0]['numClasses'])
currentModel.fc1 = new_classifier1
# fc2
new_classifier2 = nn.Linear(modelData['sizeFeatures'], classesADP[1]['numClasses'])
currentModel.fc2 = new_classifier2
# fc3
new_classifier3 = nn.Linear(modelData['sizeFeatures'], classesADP[2]['numClasses'])
currentModel.fc3 = new_classifier3

dirPretrainedModel = dirPretrainedModels + trainMode + '/' + modelData['name'] + '/'
currentModel.load_state_dict(torch.load(os.path.join(dirPretrainedModel, 'modelsave_1_final.pt')))

del currentModel.fc1
del currentModel.fc2
del currentModel.fc3

# block parameters
for param in currentModel.parameters():
    param.requires_grad = False  # frozen for warmup

# change last layer
new_fc = nn.Linear(modelData['sizeFeatures'], classesALL['numClasses'])
currentModel.fc = new_fc
imageSize = 224
currentModel.to('cuda')

**Normalization**

Let's first compute mean and std of the patches.

To do so, we need to define transforms.

In [ ]:
import PIL

# preprocess
transform = {
    'train':
        transforms.Compose([
            transforms.CenterCrop(256),
            transforms.Resize(imageSize, interpolation=PIL.Image.Resampling.BILINEAR),
            #transforms.RandomRotation(45),
            transforms.ToTensor()
        ]),
    'val':
        transforms.Compose([  # [1]
            transforms.CenterCrop(256),
            transforms.Resize(imageSize, interpolation=PIL.Image.Resampling.BILINEAR),
            transforms.ToTensor()
        ])
}

Define the dataloaders.

In [ ]:
# params
batch_sizeP_norm = 4096
numWorkersP = 0

all_idb2_train = datasets.ImageFolder(os.path.join(dirOutTrainTest, 'train'), transform['train'])
all_idb2_train_loader = torch.utils.data.DataLoader(all_idb2_train,
                                                    batch_size=batch_sizeP_norm, shuffle=False,
                                                    num_workers=numWorkersP, pin_memory=True)
print("Classi: {0}".format(all_idb2_train.classes))
dataset_sizes['train'] = len(all_idb2_train)
print("Dimensione dataset train: {0}".format(dataset_sizes['train']))

# val
all_idb2_val = datasets.ImageFolder(os.path.join(dirOutTrainTest, 'val'), transform=transform['val'])
all_idb2_val_loader = torch.utils.data.DataLoader(all_idb2_val,
                                                  batch_size=batch_sizeP_norm, shuffle=False,
                                                  num_workers=numWorkersP, pin_memory=True)
print("Classi: {0}".format(all_idb2_val.classes))
dataset_sizes['val'] = len(all_idb2_val)
print("Dimensione dataset val: {0}".format(dataset_sizes['val']))

Compute mean and std.

In [ ]:
# mean and std
print('Computing mean and standard deviation...')
meanNorm, stdNorm = util.computeMeanStd(all_idb2_train_loader, cuda, stats_path=dirPatches/"train_norm.pt")

print(meanNorm, stdNorm)

Update the transforms and add some data augmentation.

In [ ]:
transform['train'] = transforms.Compose([
    transforms.CenterCrop(256),
    transforms.Resize(imageSize, interpolation=PIL.Image.Resampling.BILINEAR),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    # transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    # transforms.RandomAffine(degrees=20, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=meanNorm,
        std=stdNorm)])
# val
transform['val'] = transforms.Compose([
    transforms.CenterCrop(256),
    transforms.Resize(imageSize, interpolation=PIL.Image.Resampling.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=meanNorm,
        std=stdNorm)])

Update the data loaders.

In [ ]:
# update data loaders
batch_sizeP = 128
numWorkersP = 4
persistent_workers = numWorkersP > 0
prefetch_factor = 1 if numWorkersP > 0 else None
# train
all_idb2_train = util.ImageFolderWithPaths(os.path.join(dirOutTrainTest, 'train'),
                                           transformP=transform['train'],
                                           classesP=classVecPart['train'],
                                           filenamesP=fileNameVecPart['train'])
all_idb2_train_loader = torch.utils.data.DataLoader(all_idb2_train,
                                                    batch_size=batch_sizeP, shuffle=True,
                                                    num_workers=numWorkersP, pin_memory=True, 
                                                    persistent_workers=persistent_workers, prefetch_factor=prefetch_factor)
# val
all_idb2_val = util.ImageFolderWithPaths(os.path.join(dirOutTrainTest, 'val'),
                                         transformP=transform['val'],
                                         classesP=classVecPart['val'],
                                         filenamesP=fileNameVecPart['val'])
all_idb2_val_loader = torch.utils.data.DataLoader(all_idb2_val,
                                                  batch_size=batch_sizeP, shuffle=False,
                                                  num_workers=numWorkersP, pin_memory=True,
                                                  persistent_workers=persistent_workers, prefetch_factor=prefetch_factor)

Define the criterion and the optimizer. We can add a scheduler also.

We have labels that are not mutually exclusive so we need something different than standard cross-entropy.

<img src="imgs/mlloss.png" alt="Alt text" width="700">

Let's do a warm-up first, to train only the last layer.

In [ ]:
"""
import importlib
importlib.invalidate_caches()
importlib.reload(functions)

import inspect
print(util.__file__)
print(inspect.signature(util.computeClassWeights))

import os, sys
print(os.getcwd())
print(sys.path[:5])
"""

In [ ]:
# class weights
print("Computing class weights...")
weightsBCE = util.computeClassWeights(all_idb2_train_loader, cuda=True, weights_path=dirPatches/"all_idb2_class_weights.pt")
# warm-up
print("Warm-up")
criterion_warmup = nn.BCEWithLogitsLoss(pos_weight=weightsBCE.float().to("cuda"))
currentModel = functions.warmup(
    currentModel,
    all_idb2_train_loader,
    log=log,
    cuda=cuda,
    dirResults=dirPatches,
    iteration=r,
    criterion=criterion_warmup,
    num_epochs_warmup=5,
    lr_warmup=1e-4,
    weight_decay=5e-4,
)

# re-enable grad
for param in currentModel.parameters():
    param.requires_grad = True  # re-enable gradients

We can now train our model.

In [ ]:
# params
lr = 1e-4
lr_min = 1e-6
num_epochs = 100
r_orth = 0.1

# criterion = torch.nn.MultiLabelSoftMarginLoss(weight=weightsBCE.float().to('cuda'))
criterion = nn.BCEWithLogitsLoss(pos_weight=weightsBCE.float().to('cuda'))
# optimizer_ft = optim.SGD(currentModel.parameters(), lr=lr, momentum=0.9, weight_decay=0.0005)
optimizer_ft = optim.AdamW(currentModel.parameters(), lr=lr, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_ft,
    T_max=num_epochs,
    eta_min=lr_min
)

print("Training...")
currentModel = functions.train_model_val(currentModel, classVecPart,
                                         optimizer_ft, scheduler, criterion,
                                         num_epochs, dataset_sizes,
                                         all_idb2_train_loader, all_idb2_val_loader,
                                         batch_sizeP, classesALL['numClasses'], modelData['name'],
                                         dirPatches, r, r_orth, log, cuda,
                                         checkpoint_every=10)

**Test**

Let's now apply the model on the test subset.

First, let's define the transforms and dataloaders.

In [ ]:
print("Testing")

# eval
currentModel.eval()
# zero the parameter gradients
optimizer_ft.zero_grad()
torch.no_grad()

# test transform
transform['test'] = transforms.Compose([
    transforms.CenterCrop(256),
    transforms.Resize(imageSize, interpolation=PIL.Image.Resampling.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=meanNorm,
        std=stdNorm),
])

# load data
all_idb2_test = util.ImageFolderWithPaths(os.path.join(dirOutTrainTest, 'test'),
                                         transformP=transform['test'],
                                         classesP=classVecPart['test'],
                                         filenamesP=fileNameVecPart['test'])
all_idb2_test_loader = torch.utils.data.DataLoader(all_idb2_test,
                                                  batch_size=batch_sizeP, shuffle=False,
                                                  num_workers=numWorkersP, pin_memory=True)
dataset_sizes['test'] = len(all_idb2_test)
print("\tDimensione dataset test: {0}".format(dataset_sizes['test']))
numBatches = {}
numBatches['test'] = np.round(dataset_sizes['test'] / batch_sizeP)

Let's now tune the thresholds for detection

In [ ]:
def collect_probs_and_labels(model, dataloader, dataset_size, num_classes, batch_sizeP, cuda):
    device = torch.device("cuda" if cuda and torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    probs_all = torch.zeros(dataset_size, num_classes)
    labels_all = torch.zeros(dataset_size, num_classes)

    with torch.no_grad():
        for batch_num, (inputs, dummyTargets, filename, y) in enumerate(dataloader):
            sizeCurrentBatch = y.size(0)

            indStart = batch_num * batch_sizeP
            indEnd = indStart + sizeCurrentBatch

            inputs = inputs.to(device, non_blocking=True)

            outputs = model(inputs)
            probs = torch.sigmoid(outputs).cpu()

            probs_all[indStart:indEnd, :] = probs
            labels_all[indStart:indEnd, :] = y.float()

    return probs_all, labels_all


def hamming_accuracy(preds, labels):
    return (preds == labels.bool()).float().mean().item()


def jaccard_score_multilabel(preds, labels, eps=1e-8):
    preds = preds.bool()
    labels = labels.bool()

    intersection = (preds & labels).sum(dim=1).float()
    union = (preds | labels).sum(dim=1).float()

    score = torch.where(
        union > 0,
        intersection / (union + eps),
        torch.ones_like(union)
    )

    return score.mean().item()


def tune_global_threshold(probs_val, labels_val, metric="jaccard"):
    thresholds = torch.linspace(0.05, 0.95, steps=91)

    best_threshold = 0.5
    best_score = -1.0

    for t in thresholds:
        preds = probs_val > t

        if metric == "jaccard":
            score = jaccard_score_multilabel(preds, labels_val)
        elif metric == "hamming":
            score = hamming_accuracy(preds, labels_val)
        else:
            raise ValueError("metric must be 'jaccard' or 'hamming'")

        if score > best_score:
            best_score = score
            best_threshold = float(t)

    return best_threshold, best_score

In [ ]:
"""
import gc
gc.collect()

del optimizer_ft
del scheduler

torch.cuda.empty_cache()
torch.cuda.ipc_collect()
"""

In [ ]:
num_workers=0
shuffle=False
pin_memory=True

val_loader_threshold = torch.utils.data.DataLoader(
    all_idb2_val,
    batch_size=batch_sizeP,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

probs_val, labels_val = collect_probs_and_labels(
    currentModel,
    val_loader_threshold,
    dataset_sizes["val"],
    classesALL["numClasses"],
    batch_sizeP,
    cuda
)

best_threshold, best_val_score = tune_global_threshold(
    probs_val,
    labels_val,
    metric="jaccard"
)

print("Best global threshold:", best_threshold)
print("Best validation Jaccard:", best_val_score)

We can now loop on the images and store the results.

Then, we compute the accuracy.

In [ ]:
from sklearn.metrics import classification_report, multilabel_confusion_matrix

# loop on images
print('Testing...')
# init
predALL_test = torch.zeros(dataset_sizes['test'], classesALL['numClasses'])
labelsALL_test = torch.zeros(dataset_sizes['test'], classesALL['numClasses'])
for batch_num, (inputs, dummyTargets, filename, y) in enumerate(all_idb2_test_loader):

    # get size of current batch
    sizeCurrentBatch = y.size(0)
    # init tensor for labels
    labelsTens = torch.zeros(sizeCurrentBatch, classesALL['numClasses'], dtype=torch.float32)

    if (batch_num+1) % 10 == 0:
        print("\tBatch n. {0} / {1}".format(batch_num+1, int(numBatches['test'])))

    # stack
    indStart = batch_num * batch_sizeP
    indEnd = indStart + sizeCurrentBatch

    # extract features
    if cuda:
        inputs = inputs.to('cuda', non_blocking=True)
        labelsTens = labelsTens.to('cuda', non_blocking=True)

    # predict
    with torch.set_grad_enabled(False):
        outputs = currentModel(inputs)
        if cuda:
            outputs = outputs.to('cuda')

        # no softmax but sigmoid
        preds = (torch.sigmoid(outputs) > best_threshold)

        predALL_test[indStart:indEnd, :] = preds
        labelsALL_test[indStart:indEnd, :] = y

# end for x,y

# accuracy
accuracyResult = torch.sum(predALL_test == labelsALL_test).double()
accuracyResult = accuracyResult / (dataset_sizes['test'] * classesALL['numClasses'])
print("Accuracy: {0:.2f}%".format(accuracyResult * 100))

# print(multilabel_confusion_matrix(labelsALL_test.numpy(), predALL_test.numpy()))


Let's plot some results.

In [ ]:
from pathlib import Path
from typing import Optional, Sequence, Tuple, Union, Callable, Dict, Any, List

import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np


@torch.inference_mode()
def predict_and_plot_multilabel(
    model: torch.nn.Module,
    image: Union[str, Path, Image.Image],
    *,
    device: Optional[Union[str, torch.device]] = None,
    transform: Optional[Callable[[Image.Image], torch.Tensor]] = None,
    class_names: Optional[Sequence[str]] = None,
    threshold: float = 0.5,
    topk: Optional[int] = None,  # show top-k by probability (even if below threshold)
    figsize: Tuple[int, int] = (6, 6),
    title_prefix: str = "Predicted labels",
    return_details: bool = True,
) -> Dict[str, Any]:
    """
    Multi-label prediction for a single image.

    Assumes model output is logits with shape:
      - [B, C] (most common) OR
      - [C] (single image without batch) OR
      - [B, C, ...] (we'll flatten after batch if needed)

    Uses sigmoid(logits) per class and thresholds independently.

    Returns:
        dict with keys:
          - probs: list[float] length C
          - pred_mask: list[int] length C (0/1)
          - active: list of (label, prob) above threshold
          - topk: optional list of (label, prob) top-k
    """
    model.eval()

    # device
    if device is None:
        try:
            device = next(model.parameters()).device
        except StopIteration:
            device = torch.device("cpu")
    else:
        device = torch.device(device)

    model = model.to(device)

    # load image
    if isinstance(image, (str, Path)):
        img = Image.open(str(image)).convert("RGB")
        img_path = str(image)
    elif isinstance(image, Image.Image):
        img = image.convert("RGB")
        img_path = "<PIL.Image>"
    else:
        raise TypeError("image must be a path or PIL.Image.Image")

    # transform
    if transform is None:
        # Minimal default: convert to [0,1] tensor CHW
        img_np = np.asarray(img).astype(np.float32) / 255.0
        x = torch.from_numpy(img_np).permute(2, 0, 1)  # C,H,W
    else:
        x = transform(img)
        if not torch.is_tensor(x):
            raise ValueError("transform must return a torch.Tensor (C,H,W).")

    x = x.unsqueeze(0).to(device)  # [1,C,H,W]

    # forward
    logits = model(x)

    # normalize logits shape to [1, C]
    if isinstance(logits, (tuple, list)):
        # some models return (logits, aux); take first
        logits = logits[0]

    if logits.ndim == 1:
        logits = logits.unsqueeze(0)  # [1,C]
    elif logits.ndim > 2:
        # If model outputs spatial map, flatten everything but batch
        logits = logits.view(logits.shape[0], -1)

    if logits.shape[0] != 1:
        raise ValueError(f"Expected batch size 1, got logits shape {tuple(logits.shape)}")

    C = logits.shape[1]

    # class names
    if class_names is None:
        class_names = [f"class_{i}" for i in range(C)]
    if len(class_names) != C:
        # allow mismatch but keep safe
        class_names = list(class_names) + [f"class_{i}" for i in range(len(class_names), C)]

    # sigmoid probabilities
    probs_t = torch.sigmoid(logits)[0]  # [C]
    probs = probs_t.detach().cpu().tolist()

    # threshold
    pred_mask = [int(p >= threshold) for p in probs]

    active = [(class_names[i], probs[i]) for i in range(C) if pred_mask[i] == 1]
    active_sorted = sorted(active, key=lambda x: x[1], reverse=True)

    # top-k (optional)
    topk_info: Optional[List[Tuple[str, float]]] = None
    if topk is not None:
        k = max(1, min(int(topk), C))
        idx_sorted = np.argsort(probs)[::-1][:k]
        topk_info = [(class_names[i], probs[i]) for i in idx_sorted]

    # build title text
    if active_sorted:
        lines = [f"{lab}: {p:.2f}" for lab, p in active_sorted]
        title = f"{title_prefix} (thr={threshold}):\n" + ", ".join(lines[:8])
        if len(lines) > 8:
            title += f"\n(+{len(lines)-8} more)"
    else:
        title = f"{title_prefix}: none (thr={threshold})"

    if topk_info is not None:
        top_str = ", ".join([f"{lab}:{p:.2f}" for lab, p in topk_info])
        title += f"\n{top_str}"

    # plot
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis("off")
    plt.title(title)
    plt.show()

    if not return_details:
        return {}

    return {
        "image": img_path,
        "threshold": threshold,
        "probs": probs,
        "pred_mask": pred_mask,
        "active": active_sorted,
        "topk": topk_info,
        "device": str(device),
        "num_classes": C,
    }


The patch Im001_1_patch_28.tif proves an interesting example since it contains both a normal WBC and a blast.

In [ ]:
img = Image.open(dirPatches / 'Im001_1_patch_28.tif')
# img = Image.open(dirPatches / 'Im006_1_patch_15.tif')

out = predict_and_plot_multilabel(
    currentModel,
    img,
    device="cuda",
    transform=transform['test'],
    class_names=["blast", "normal"],
    threshold=best_threshold,
    topk=3
)

**Grad-CAM**

Let's visualize what areas correspond to each class.

In [ ]:
import copy
import torch
import matplotlib.pyplot as plt
from torchvision import transforms
from gradcam.utils import visualize_cam
from gradcam import GradCAM


def plot_gradcam_multilabel(
    model,
    img,
    mean_norm,
    std_norm,
    image_size,
    class_indices=(0, 1),
    class_names=None,
    target_layer_name="layer4",
    device="cpu",
    center_crop_size=256,
    figsize_per_class=4,
):
    """
    Plot GradCAM visualizations for a multi-label model.

    Parameters
    ----------
    model : torch.nn.Module
        Trained PyTorch model.
    img : PIL.Image
        Input image before transforms.
    mean_norm : list/tuple
        Normalization mean used during training.
    std_norm : list/tuple
        Normalization std used during training.
    image_size : int or tuple
        Resize target used by the model, e.g. 224 or (224, 224).
    class_indices : tuple/list
        Class indices for which GradCAM should be computed.
    class_names : list/dict/None
        Names for classes. If list, class_names[class_idx] is used.
        If dict, class_names[class_idx] is used.
    target_layer_name : str
        Name of the layer to use for GradCAM, e.g. "layer4".
    device : str
        Usually "cpu" for your GradCAM package, or "cuda" if supported.
    center_crop_size : int
        Center crop before resize.
    figsize_per_class : int/float
        Width per class in the figure.
    """

    # Deep copy avoids modifying the original model/device state
    model_gcam = copy.deepcopy(model)
    model_gcam.to(device)
    model_gcam.eval()

    # Get target layer by name
    target_layer = getattr(model_gcam, target_layer_name)

    gradcam = GradCAM(model_gcam, target_layer)

    # Non-normalized image for visualization
    transform_gradcam = transforms.Compose([
        transforms.CenterCrop(center_crop_size),
        transforms.Resize(image_size, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.ToTensor(),
    ])

    img_tensor = transform_gradcam(img)

    # Normalized image for model input
    img_tensor_norm = transforms.Normalize(mean_norm, std_norm)(img_tensor)
    img_tensor_norm = img_tensor_norm.unsqueeze(0).to(device)  # [1, C, H, W]

    results = {}

    n_classes = len(class_indices)
    fig, axes = plt.subplots(
        1,
        n_classes,
        figsize=(figsize_per_class * n_classes, figsize_per_class)
    )

    if n_classes == 1:
        axes = [axes]

    heatmaps = []

    for ax, class_idx in zip(axes, class_indices):
        mask, _ = gradcam(img_tensor_norm, class_idx=class_idx)

        heatmap, result = visualize_cam(mask, img_tensor)
        heatmaps.append(heatmap)

        img_plot = result.detach().cpu()
        img_plot = img_plot.permute(1, 2, 0).numpy()

        results[class_idx] = {
            "mask": mask.detach().cpu(),
            "heatmap": heatmap.detach().cpu(),
            "overlay": img_plot,
        }

        if class_names is None:
            title = f"Class {class_idx}"
        else:
            title = f"Class {class_idx}: {class_names[class_idx]}"

        ax.imshow(img_plot)
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    return results, transform_gradcam, heatmaps

In [ ]:
results, transform_gradcam, heatmaps = plot_gradcam_multilabel(
    model=currentModel,
    img=img,
    mean_norm=meanNorm,
    std_norm=stdNorm,
    image_size=imageSize,
    class_indices=(0, 1),
    class_names={0: "blast", 1: "normal"},
    target_layer_name="layer4",
    device="cpu",
    center_crop_size=256,
)

## Part 3. Back to WSIs

Let's try to go back to WSI. Specifically, we would like the WSI with the markup of knowing where the blasts are.

First, we need to define some functions that allows us to go from the Grad-CAM heatmap to a centroid. We need to:
* Convert a JET color scheme into a 2-D intensity image;
* Extract an intensity-weighted centroid.

In [ ]:
from matplotlib import pyplot
from skimage import measure, filters

def jet_rgb_to_scalar(heatmap_rgb, resolution=256):
    """
    Convert an RGB heatmap using JET colormap back to a 2D scalar map in [0,1].

    heatmap_rgb: torch.Tensor or np.ndarray, shape (3,H,W) or (H,W,3), values in [0,1]
    resolution: number of samples used to invert the colormap

    Returns:
        scalar_map: np.ndarray shape (H,W), values in [0,1]
    """
    # Convert to numpy HWC
    if torch.is_tensor(heatmap_rgb):
        heatmap_rgb = heatmap_rgb.detach().cpu().numpy()

    if heatmap_rgb.shape[0] == 3:
        heatmap_rgb = np.transpose(heatmap_rgb, (1, 2, 0))

    heatmap_rgb = heatmap_rgb.astype(np.float32)

    # Build JET lookup table
    jet = pyplot.get_cmap("jet", resolution)
    jet_colors = jet(np.linspace(0, 1, resolution))[:, :3]  # (R,3)

    # Flatten image
    H, W, _ = heatmap_rgb.shape
    flat = heatmap_rgb.reshape(-1, 3)

    # Compute distance to each JET color
    # Result shape: (H*W, resolution)
    dists = np.linalg.norm(flat[:, None, :] - jet_colors[None, :, :], axis=2)

    # Best matching JET index
    idx = np.argmin(dists, axis=1)

    # Normalize index → [0,1]
    scalar = idx.astype(np.float32) / (resolution - 1)
    scalar_map = scalar.reshape(H, W)

    return scalar_map

def normalize_heatmap(hm: np.ndarray):
    """Scale heatmap to [0,1] (in-place safe: returns new array)."""
    hm = np.asarray(hm, dtype=float)
    if hm.size == 0:
        return hm
    mn, mx = hm.min(), hm.max()
    if mx <= mn:
        return np.zeros_like(hm)
    return (hm - mn) / (mx - mn)

def intensity_weighted_centroid(hm: np.ndarray):
    """
    Return (x, y) centroid using intensity weights (x = column, y = row).
    If sum of weights is 0 returns None.
    """
    hm = normalize_heatmap(hm)
    total = hm.sum()
    if total == 0:
        return None
    # coordinates arrays: y indices (rows) and x indices (cols)
    rows = np.arange(hm.shape[0], dtype=float)
    cols = np.arange(hm.shape[1], dtype=float)
    # compute weighted means
    cy = (hm * rows[:, None]).sum() / total   # row (y)
    cx = (hm * cols[None, :]).sum() / total   # column (x)
    return float(cx), float(cy)

def overlay_centroid_on_image(image, centroid, marker='x', color='red', size=100, show=True, figsize=(6,6), title=''):
    """
    image: PIL.Image or numpy array (H,W,C) or (H,W).
    centroid: (x, y) in pixel coordinates (x=col, y=row)
    """
    import matplotlib.pyplot as plt
    from PIL import Image
    if isinstance(image, Image.Image):
        img = np.asarray(image)
    else:
        img = np.asarray(image)

    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img)
    if centroid is not None:
        cx, cy = centroid
        ax.scatter([cx], [cy], marker=marker, c=color, s=size)
        ax.text(cx + 5, cy - 5, f"({cx:.1f},{cy:.1f})", color=color, bbox=dict(facecolor='white', alpha=0.6, pad=1), fontsize=8)
    ax.axis('off')
    if show:
        plt.title(title)
        plt.show()
    return fig, ax

def to_2d_heatmap(hm, method='mean'):
    """
    Convert heatmap to 2D single-channel.
    hm: torch.Tensor or np.ndarray with shape (3,H,W) or (H,W,3)
    method: 'mean' | 'max' | 'lum' (luminance: 0.299R+0.587G+0.114B)
    Returns: 2D numpy array float32 (H,W)
    """
    if torch.is_tensor(hm):
        hm = hm.detach().cpu().numpy()
    hm = np.asarray(hm)

    # Normalize shape to (3,H,W)
    if hm.ndim == 3 and hm.shape[0] != 3 and hm.shape[2] == 3:
        # shape (H,W,3) -> (3,H,W)
        hm = np.transpose(hm, (2,0,1))

    if hm.ndim != 3 or hm.shape[0] != 3:
        raise ValueError(f"expected 3-channel heatmap, got shape {hm.shape}")

    # convert to float
    hm = hm.astype(np.float32)

    if method == 'mean':
        out = hm.mean(axis=0)
    elif method == 'max':
        out = hm.max(axis=0)
    elif method == 'lum':
        # hm[0]=R, hm[1]=G, hm[2]=B
        out = 0.299*hm[0] + 0.587*hm[1] + 0.114*hm[2]
    else:
        raise ValueError("method must be 'mean', 'max', or 'lum'")

    # normalize to [0,1]
    mn, mx = out.min(), out.max()
    if mx > mn:
        out = (out - mn) / (mx - mn)
    else:
        out = np.zeros_like(out)
    return out

def keep_largest_cc_heatmap(hm2d, threshold=None, threshold_mode="otsu", connectivity=2):
    """
    hm2d: 2D numpy array (float)
    threshold: float or None
    threshold_mode:
        - "otsu" (default)
        - "mean"
        - "percentile:80"  (keep top 20%)
    connectivity: 1 (4-neigh) or 2 (8-neigh)

    Returns:
        masked_hm: same shape as hm2d, values preserved only in largest CC, else 0
        mask_largest: boolean mask of largest CC
        thr: threshold used
    """
    hm = np.asarray(hm2d, dtype=float)

    # choose threshold
    if threshold is None:
        if threshold_mode == "otsu":
            thr = float(filters.threshold_otsu(hm))
        elif threshold_mode == "mean":
            thr = float(hm.mean())
        elif threshold_mode.startswith("percentile:"):
            p = float(threshold_mode.split(":")[1])
            thr = float(np.percentile(hm, p))
        else:
            raise ValueError("threshold_mode must be 'otsu', 'mean', or 'percentile:XX'")
    else:
        thr = float(threshold)

    mask = hm >= thr
    if mask.sum() == 0:
        return np.zeros_like(hm), np.zeros_like(mask, dtype=bool), thr

    lbl = measure.label(mask, connectivity=connectivity)
    props = measure.regionprops(lbl)
    if not props:
        return np.zeros_like(hm), np.zeros_like(mask, dtype=bool), thr

    largest = max(props, key=lambda p: p.area)
    mask_largest = (lbl == largest.label)

    masked_hm = np.zeros_like(hm)
    masked_hm[mask_largest] = hm[mask_largest]
    return masked_hm, mask_largest, thr

Let's plot an example.

In [ ]:
# JET to intensity.
scalar_map = normalize_heatmap(jet_rgb_to_scalar(heatmaps[0]))
# only largest cc
masked_hm, _, _ = keep_largest_cc_heatmap(scalar_map, threshold=0.8)
# plt.imshow(masked_hm)
# intensity-weighted centroid
c_w = intensity_weighted_centroid(masked_hm)
# print("Intensity-weighted centroid (x,y):", c_w)
# display
_ = overlay_centroid_on_image(img, c_w, title='Is there a blast in this patch?')

Let's do that for all patches. We can directly compute the centroids within the original WSI.

In [ ]:
def patch_centroid_to_wsi(
    cx_patch: float,
    cy_patch: float,
    patch_info: PatchInfo,
    *,
    one_based_patch: bool = True,
) -> Tuple[float, float]:
    """
    Convert centroid in patch coordinates to coordinates in the original WSI.

    Args:
        cx_patch: x (column) inside patch
        cy_patch: y (row) inside patch
        patch_info: PatchInfo with Start = (row_start, col_start) in WSI
        one_based_patch: True if patch coords are 1-based (MATLAB),
                         False if 0-based (Python/NumPy)

    Returns:
        (cx_wsi, cy_wsi): centroid coordinates in WSI (x = col, y = row)
    """
    row_start, col_start = patch_info.Start
    offset = 1 if one_based_patch else 0

    cy_wsi = row_start + (cy_patch - offset)
    cx_wsi = col_start + (cx_patch - offset)

    return cx_wsi, cy_wsi

In [ ]:
# dirs
dirBlastCoord = dirPatches / 'blast_positions'
os.makedirs(dirBlastCoord, exist_ok=True)

In [ ]:
# long!
# extract list of files
extPatch = 'tif'
files = sorted(dirPatches.glob(f"*.{extPatch}"))

if 0:
    # loop
    for num, img_path in enumerate(files):

        # if num > 200:
            # break

        patch_name = img_path.stem
        wsi_name = patch_name.split("_patch_")[0]
        wsi_blast_coord_info_name = dirBlastCoord / Path(wsi_name + ".csv")

        # If output CSV for this WSI already exists, skip this patch
        if wsi_blast_coord_info_name.is_file():
            print(f"Skipping {patch_name}: {wsi_blast_coord_info_name.name} already exists")
            continue

        img = Image.open(img_path)

        # create a non-normalized and a normalized version
        imgTensor = transform_gradcam(img)
        imgTensorNorm = transforms.Normalize(meanNorm, stdNorm)(imgTensor)[None]
        imgTensorNorm = imgTensorNorm.squeeze().unsqueeze(0).to('cpu')  # [1,C,H,W]

        # class 0
        # is there?
        with torch.no_grad():
            output = currentModel(imgTensorNorm.to('cuda'))

        if torch.sigmoid(output)[0, 0] > best_threshold:  # it's there
            mask, _ = gradcam(imgTensorNorm, class_idx=0)
            heatmap, result = visualize_cam(mask, imgTensor)

            # JET to intensity
            scalar_map = normalize_heatmap(jet_rgb_to_scalar(heatmap))

            # only largest cc
            masked_hm, _, _ = keep_largest_cc_heatmap(scalar_map, threshold=0.8)

            # intensity-weighted centroid
            c_w = intensity_weighted_centroid(masked_hm)

            # load info for the patch
            fileInfoPath = dirInfo / Path(img_path.stem).with_suffix(".dat")
            info = load_patchinfos_pickle(fileInfoPath)

            # heatmap shape
            H_cam, W_cam = heatmap.shape[-2], heatmap.shape[-1]

            # patch size from PatchInfo
            patch_h, patch_w = info.BlockSize

            # centroid on CAM: (cx_cam, cy_cam)
            cx_patch = c_w[0] * (patch_w / W_cam)
            cy_patch = c_w[1] * (patch_h / H_cam)

            # coordinates in the original WSI
            cx_wsi, cy_wsi = patch_centroid_to_wsi(cx_patch, cy_patch, info)

            with open(wsi_blast_coord_info_name, 'a', encoding='utf-8') as f:
                f.write(f"{wsi_name},{patch_name},{cx_wsi:.2f},{cy_wsi:.2f}\n")

            print('Patch {}, blast centroid: {}'.format(img_path, c_w))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def overlay_centroids_from_csv_noheader(
    wsi,
    csv_path: str | Path,
    *,
    marker: str = "x",
    color: str = "red",
    size: int = 60,
    alpha: float = 0.9,
    figsize=(12, 12),
    save_path: str | Path | None = None,
    title=''
):
    """
    CSV format (NO HEADER), 4 columns:
      0: wsi_id (e.g. 'Im001_1')
      1: patch_name (e.g. 'Im001_1_patch_14')
      2: cx (x / column in WSI pixels)
      3: cy (y / row in WSI pixels)
    """
    csv_path = Path(csv_path)

    # Load WSI image
    W, H = wsi.size

    # Load CSV (no header)
    df = pd.read_csv(csv_path, header=None, names=["wsi_id", "patch_name", "cx", "cy"])
    df["cx"] = pd.to_numeric(df["cx"], errors="coerce")
    df["cy"] = pd.to_numeric(df["cy"], errors="coerce")
    df = df.dropna(subset=["cx", "cy"])

    # Clamp to image bounds (optional safety)
    df["cx"] = df["cx"].clip(0, W - 1)
    df["cy"] = df["cy"].clip(0, H - 1)

    # Plot
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(wsi)
    ax.scatter(df["cx"].to_numpy(), df["cy"].to_numpy(),
               marker=marker, c=color, s=size, alpha=alpha)
    ax.set_xlim(0, W - 1)
    ax.set_ylim(H - 1, 0)  # top-left origin like images
    ax.axis("off")
    ax.set_title(f"{title}: {len(df)} points")

    plt.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches="tight", dpi=150)

    plt.show()
    return fig, ax, df

In [ ]:
# directory of WSI
dirWSIs = Path('./WSIs/')
ext = 'jpg'

# extract list of files
files = sorted(dirWSIs.glob(f"*.{ext}"))
# get the first file and open
# img_path = next(enumerate(files))
# img = Image.open(img_path[1])
img_path = Path('WSIs/Im001_1.jpg')
# img_path = Path('WSIs/Im002_1.jpg')
# img_path = Path('WSIs/Im021_1.jpg')
img = Image.open(img_path)
# get the info file name
wsi_name = img_path.stem
wsi_blast_coord_info_name = dirBlastCoord / Path(wsi_name + '.csv')
# print(wsi_blast_coord_info_name)

# display
fig, ax, df = overlay_centroids_from_csv_noheader(
    wsi=img,
    csv_path=wsi_blast_coord_info_name,
    title=wsi_name
)

Next: clean results by having at most 1 point per cell...

Problems? 
A single centroid even for patches with multiple blasts. 